In [9]:
import os
import sys
import json
import random
import warnings

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from torchvision import models

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

sys.path.append(os.path.abspath("../src"))

from dataset import ButterflyDataset
from transforms import get_transforms

Device: cpu


In [11]:
# Download / localizar dataset
path = kagglehub.competition_download('aca-butterflies')
competition_dir = os.path.join(path)
train_dir = os.path.join(competition_dir, "train")
test_dir = os.path.join(competition_dir, "test")
train_csv = os.path.join(competition_dir, "train.csv")

# Ler CSV
df = pd.read_csv(train_csv)

# Split estratificado
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

# Label encoding
classes = sorted(df["label"].unique())
class_to_idx = {cls: i for i, cls in enumerate(classes)}
idx_to_class = {i: cls for cls, i in class_to_idx.items()}

train_df["label_idx"] = train_df["label"].map(class_to_idx).astype(int)
val_df["label_idx"] = val_df["label"].map(class_to_idx).astype(int)

# Transforms
train_transform, val_transform, test_transform = get_transforms()

# Datasets
train_dataset = ButterflyDataset(train_df, train_dir, transform=train_transform)
val_dataset = ButterflyDataset(val_df, train_dir, transform=val_transform)

# DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

Train samples: 4159
Validation samples: 1040
